# Chapter 41: Deep Learning Foundations for Vision
## Multimedia & Computer Vision Learning Hub — ITERA
**Author:** Aji Pamungkas Tri Nurcahyo
**Objective:** Memahami operasi fundamental Convolutional Neural Network (CNN) mulai dari formulasi matematika cross-correlation 2D, implementasi algoritma dasar (NumPy), hingga alur kerja pelatihan model riil menggunakan PyTorch autograd engine.

### 1. Landasan Teori & Formulasi Matematika
Operasi konvolusi diskrit 2D (korelasi silang) pada citra input $I$ dengan kernel $K$:
$$S(i, j) = (I * K)(i, j) = \sum_{m=0}^{K_h-1} \sum_{n=0}^{K_w-1} I(i+m, j+n) K(m, n)$$

Dimensi spasial output dengan padding $P$ dan stride $S$:
$$O = \left\lfloor \frac{W - K + 2P}{S} \right\rfloor + 1$$

Aritmatika parameter Conv2D dengan bobot $\mathbf{W} \in \mathbb{R}^{C_{\text{out}} \times C_{\text{in}} \times K_h \times K_w}$ dan bias $\mathbf{b} \in \mathbb{R}^{C_{\text{out}}}$:
$$\mathcal{P}_{\text{conv}} = (K_h \cdot K_w \cdot C_{\text{in}} + 1) \cdot C_{\text{out}}$$

In [ ]:
# 2. Setup Lingkungan & Pengecekan Runtime (Dual-Engine Mode)
import numpy as np
import sys

print(f"Python Version: {sys.version}")
print(f"NumPy Version: {np.__version__}")

HAS_TORCH = False
try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    HAS_TORCH = True
    print(f"PyTorch Version: {torch.__version__} (Device: {'cuda' if torch.cuda.is_available() else 'cpu'})")
except ImportError:
    print("INFO: PyTorch tidak terpasang di host lokal. Kode PyTorch dirancang dengan fallback mandiri.")
    print("Untuk menjalankan pelatihan nyata pada GPU/Colab, ikuti panduan di content/notebooks-dl-setup.md")

In [ ]:
# 3. Implementasi 2D Convolution dari Scratch (NumPy Engine)
def conv2d_forward(x, kernel, bias=0.0, stride=1, padding=0):
    h_in, w_in = x.shape
    k_h, k_w = kernel.shape
    h_out = (h_in - k_h + 2 * padding) // stride + 1
    w_out = (w_in - k_w + 2 * padding) // stride + 1
    if padding > 0:
        x_pad = np.pad(x, ((padding, padding), (padding, padding)), mode='constant')
    else:
        x_pad = x
    out = np.zeros((h_out, w_out), dtype=np.float64)
    for i in range(h_out):
        for j in range(w_out):
            patch = x_pad[i*stride:i*stride+k_h, j*stride:j*stride+k_w]
            out[i, j] = np.sum(patch * kernel) + bias
    return out

# Uji coba deterministik dengan matriks 4x4 dan filter Sobel
x = np.array([
    [10, 20, 20, 10],
    [30, 50, 40, 20],
    [10, 60, 50, 30],
    [20, 10, 30, 20]
], dtype=np.float64)

kernel = np.array([
    [-1, -2, -1],
    [ 0,  0,  0],
    [ 1,  2,  1]
], dtype=np.float64)

res_numpy = conv2d_forward(x, kernel, bias=5.0)
print("Output Konvolusi 2D (NumPy):")
print(res_numpy)
assert np.isclose(res_numpy[0, 0], 115.0), "Elemen (0,0) harus 115.0"
assert np.isclose(res_numpy[0, 1], 125.0), "Elemen (0,1) harus 125.0"

In [ ]:
# 4. Fungsi Aktivasi Non-Linier: ReLU & Softmax (Analitis)
def relu(x):
    return np.maximum(0, x)

def softmax(logits):
    e = np.exp(logits - np.max(logits))
    return e / np.sum(e)

logits = np.array([2.5, 0.5, -1.0])
probs = softmax(logits)
print(f"Logits: {logits}")
print(f"Probabilitas Softmax: {np.round(probs, 4)}")
print(f"Aktivasi ReLU pada Output Konvolusi:\n{relu(res_numpy)}")

In [ ]:
# 5. Parameter Counting & Receptive Field Arithmetic
def count_params(c_in, c_out, k=3, bias=True):
    return (k * k * c_in + (1 if bias else 0)) * c_out

p1 = count_params(3, 64, k=3)
p2 = count_params(64, 128, k=3)
print(f"Conv2D(3 -> 64, k=3): {p1} parameters (Teori: 1,792)")
print(f"Conv2D(64 -> 128, k=3): {p2} parameters (Teori: 73,856)")
assert p1 == (3 * 3 * 3 + 1) * 64
assert p2 == (3 * 3 * 64 + 1) * 128

In [ ]:
# 6. Real PyTorch Deep Learning Workflow (Arsitektur Model, Autograd & Training Loop)
if HAS_TORCH:
    # Definisi arsitektur CNN riil
    class MiniConvNet(nn.Module):
        def __init__(self, num_classes=2):
            super().__init__()
            self.features = nn.Sequential(
                nn.Conv2d(1, 4, kernel_size=3, padding=1), # (B, 4, 16, 16)
                nn.BatchNorm2d(4),
                nn.ReLU(),
                nn.MaxPool2d(2, 2), # (B, 4, 8, 8)
                nn.Conv2d(4, 8, kernel_size=3, padding=1), # (B, 8, 8, 8)
                nn.ReLU(),
                nn.AdaptiveAvgPool2d((1, 1)) # (B, 8, 1, 1)
            )
            self.classifier = nn.Linear(8, num_classes)

        def forward(self, x):
            feat = self.features(x)
            flat = torch.flatten(feat, 1)
            return self.classifier(flat)

    model = MiniConvNet()
    print("PyTorch Model Architecture Initialized:")
    print(model)
    # Hitung total parameter PyTorch
    total_torch_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total Trainable Parameters in MiniConvNet: {total_torch_params}")

    # Sintesis mini batch data latih (B=8, C=1, H=16, W=16)
    torch.manual_seed(42)
    x_synthetic = torch.randn(8, 1, 16, 16)
    y_synthetic = torch.randint(0, 2, (8,))

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=0.05, momentum=0.9)

    # Jalankan training loop 3 epoch
    print("\nMenjalankan PyTorch Autograd Training Loop:")
    for epoch in range(1, 4):
        model.train()
        optimizer.zero_grad()
        out = model(x_synthetic)
        loss = criterion(out, y_synthetic)
        loss.backward()
        optimizer.step()
        print(f"Epoch {epoch}/3 - Loss: {loss.item():.4f}")
else:
    print("Simulasi Alur Autograd & Optimasi Gradien (Fallback Mode):")
    # Representasi deterministik optimasi SGD
    w_mock = 2.0
    for ep in range(1, 4):
        grad_mock = 2.0 * w_mock # d(w^2)/dw
        w_mock -= 0.1 * grad_mock
        loss_mock = w_mock**2
        print(f"Epoch {ep}/3 - Simulated SGD Step: w={w_mock:.4f}, Loss={loss_mock:.4f}")
    print("Untuk menjalankan skrip penuh PyTorch MiniConvNet, aktifkan environment PyTorch.")

### 7. Analisis Hasil, Komparasi & Kesimpulan
1. **Komparasi NumPy vs PyTorch**: Operasi korelasi silang NumPy membuktikan mekanisme matematis geser-dan-kalikan secara transparan, sementara PyTorch `nn.Conv2d` mengemas operasi tersebut ke dalam kernel C++/CUDA yang teroptimasi dengan autograd diferensiasi otomatis.
2. **Aritmatika Receptive Field & Parameter**: Penggunaan filter kecil $3 \times 3$ yang ditumpuk mampu memperluas medan reseptif secara bertahap dengan penghematan parameter yang signifikan dibandingkan filter besar tunggal.
3. **Reproduksibilitas**: Seluruh sel komputasi di atas tervalidasi mengeksekusi dengan bersih pada lingkungan CPU lokal dan siap dieskalasi ke Google Colab / GPU.